In [ ]:
!pip install -q \
    transformers>=4.47.0 \
    trl>=0.12.0 \
    peft>=0.13.0 \
    accelerate>=1.2.0 \
    bitsandbytes>=0.44.0 \
    datasets>=3.0.0 \
    torch>=2.4.0

In [ ]:
import torch

assert torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')
print(f'PyTorch: {torch.__version__}')

import transformers, trl, peft
print(f'Transformers: {transformers.__version__}  |  TRL: {trl.__version__}  |  PEFT: {peft.__version__}')

GPU: NVIDIA A100-SXM4-80GB  |  VRAM: 85.1 GB
PyTorch: 2.10.0+cu128
Transformers: 5.0.0  |  TRL: 0.29.1  |  PEFT: 0.18.1


In [ ]:
from datasets import Dataset
import json
import os



SYSTEM_PROMPT = (
    'You are an expert terminal agent. Given a task, reason step by step '
    'inside <think>...</think> tags, then output your final bash solution '
    'inside <answer>...</answer> tags. Output ONLY these two sections.'
)

TASKS_FILE = "tasks.json"

with open(TASKS_FILE, 'r') as f:
        RAW_TASKS = json.load(f)



def build_prompt(instruction: str) -> list:
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': instruction},
    ]


records = [
    {
        'prompt':      build_prompt(t['instruction']),
        'reference':   t['reference'],
        'test_script': t['test_script'],
        'difficulty':  t['difficulty'],
        'category':    t['category'],
    }
    for t in RAW_TASKS
]

split_idx     = int(len(records) * 0.8)
train_dataset = Dataset.from_list(records[:split_idx])
eval_dataset  = Dataset.from_list(records[split_idx:])

print(f'Dataset built, Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

Dataset built, Train: 13 | Eval: 4


In [ ]:
import re
import subprocess
import tempfile
import os
from typing import List


def get_completion_text(comp) -> str:
    """
    Normalise a completion to a plain string.

    TRL >=0.15 may pass completions as:
      - str                        (older behaviour)
      - list[dict]  (chat messages – take the last assistant turn)
      - list[str]   (multiple string chunks – join them)
    """
    if isinstance(comp, str):
        return comp
    if isinstance(comp, list):
        # list of message dicts  →  extract last assistant content
        for msg in reversed(comp):
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                content = msg.get("content", "")
                return content if isinstance(content, str) else " ".join(content)
        # list of strings  →  join
        return " ".join(c if isinstance(c, str) else str(c) for c in comp)
    return str(comp)


def extract_answer(text: str) -> str:
    """Pull content from <answer>...</answer>. Falls back to full text."""
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()


def format_reward(completions: List, **kwargs) -> List[float]:
    """0.1 if both <think> and <answer> tags are present, else 0.0."""
    rewards = []
    for comp in completions:
        text = get_completion_text(comp)
        has_think  = bool(re.search(r'<think>.*?</think>',   text, re.DOTALL))
        has_answer = bool(re.search(r'<answer>.*?</answer>', text, re.DOTALL))
        rewards.append(0.1 if (has_think and has_answer) else 0.0)
    return rewards


def task_pass_reward(completions: List, test_script: List[str], **kwargs) -> List[float]:
    """
    1.0 if the generated bash script passes the test_script check, else 0.0.
    Runs in an isolated temp directory with a 10-second timeout.
    In production, replace subprocess with the TerminalBench2 Docker harness.
    """
    rewards = []
    for comp, test in zip(completions, test_script):
        answer = extract_answer(get_completion_text(comp))
        if not answer:
            rewards.append(0.0)
            continue
        with tempfile.TemporaryDirectory() as tmpdir:
            agent_path = os.path.join(tmpdir, 'agent.sh')
            test_path  = os.path.join(tmpdir, 'test.sh')
            with open(agent_path, 'w') as f:
                f.write('#!/bin/bash\nset -e\n' + answer + '\n')
            with open(test_path, 'w') as f:
                f.write('#!/bin/bash\n' + test + '\n')
            os.chmod(agent_path, 0o755)
            os.chmod(test_path,  0o755)
            try:
                run = subprocess.run(
                    ['bash', agent_path],
                    capture_output=True, text=True, timeout=10, cwd=tmpdir
                )
                if run.returncode != 0:
                    rewards.append(0.0)
                    continue
                chk = subprocess.run(
                    ['bash', test_path],
                    capture_output=True, text=True, timeout=5, cwd=tmpdir
                )
                rewards.append(1.0 if chk.returncode == 0 else 0.0)
            except subprocess.TimeoutExpired:
                rewards.append(0.0)
            except Exception:
                rewards.append(0.0)
    return rewards


def length_penalty_reward(completions: List, **kwargs) -> List[float]:
    """-0.2 if completion exceeds ~800 tokens (approx 3200 chars), else 0.0."""
    MAX_CHARS = 800 * 4
    return [-0.2 if len(get_completion_text(c)) > MAX_CHARS else 0.0 for c in completions]


# ── Smoke-test the reward functions ──────────────────────────────────────────
_good = '<think>step</think><answer>echo hi</answer>'
_bad  = 'just some text'

# Test with plain strings (old TRL style)
assert format_reward([_good]) == [0.1], "format_reward plain string failed"
assert format_reward([_bad])  == [0.0], "format_reward plain string bad failed"

# Test with list-of-dicts (new TRL style)
_good_msg = [{"role": "assistant", "content": _good}]
_bad_msg  = [{"role": "assistant", "content": _bad}]
assert format_reward([_good_msg]) == [0.1], "format_reward message dict failed"
assert format_reward([_bad_msg])  == [0.0], "format_reward message dict bad failed"

assert task_pass_reward([_good], test_script=['echo PASS']) == [1.0]
assert task_pass_reward([_bad],  test_script=['exit 1'])    == [0.0]
assert task_pass_reward([_good_msg], test_script=['echo PASS']) == [1.0]

assert length_penalty_reward([_good])     == [0.0]
assert length_penalty_reward(['x' * 3201]) == [-0.2]
assert length_penalty_reward([_good_msg]) == [0.0]

print("format_reward         (plain string, good):", format_reward([_good]))
print("format_reward         (plain string, bad): ", format_reward([_bad]))
print("format_reward         (message dict, good):", format_reward([_good_msg]))
print("task_pass_reward      (good):", task_pass_reward([_good], test_script=['echo PASS']))
print("task_pass_reward      (bad): ", task_pass_reward([_bad],  test_script=['exit 1']))
print("task_pass_reward      (msg dict, good):", task_pass_reward([_good_msg], test_script=['echo PASS']))
print("length_penalty_reward (short):", length_penalty_reward([_good]))
print("length_penalty_reward (long): ", length_penalty_reward(['x' * 3201]))
print("\nAll reward function smoke-tests passed ✓")


format_reward         (plain string, good): [0.1]
format_reward         (plain string, bad):  [0.0]
format_reward         (message dict, good): [0.1]
task_pass_reward      (good): [1.0]
task_pass_reward      (bad):  [0.0]
task_pass_reward      (msg dict, good): [1.0]
length_penalty_reward (short): [0.0]
length_penalty_reward (long):  [-0.2]

All reward function smoke-tests passed ✓


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

print(f'Loading model in bf16...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'\nModel ready — VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

Loading tokenizer...
Loading model in bf16...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273

Model ready — VRAM: 46.2 GB


In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = '/content/rlvr_terminalbench2'

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,


    learning_rate=5e-6,
    optim='adamw_torch',
    adam_beta1=0.9,
    adam_beta2=0.999,
    weight_decay=0.01,


    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    max_steps=500,
    warmup_steps=20,


    num_generations=8,
    max_completion_length=512,
    temperature=0.9,


    bf16=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,


    logging_steps=5,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_dataset,
    reward_funcs=[
        task_pass_reward,
        format_reward,
        length_penalty_reward,
    ],
)

print('GRPOTrainer ready')
print(f'Effective batch size : {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps}')
print(f'Completions per step : {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps * grpo_config.num_generations}')

GRPOTrainer ready
Effective batch size : 16
Completions per step : 128


In [ ]:
print('Starting GRPO training')
train_result = trainer.train()
print(f'\nTraining complete — final loss: {train_result.training_loss:.4f}')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting GRPO training


Step,Training Loss
5,0.000273
10,0.041831
15,0.005573
20,0.010937
25,0.001385
30,-0.019827
35,0.016497
40,0.023691
45,-0.011371
50,0.030723


Step,Training Loss
5,0.000273
10,0.041831
15,0.005573
20,0.010937
25,0.001385
30,-0.019827
35,0.016497
40,0.023691
45,-0.011371
50,0.030723



Training complete — final loss: 0.0102

Training complete — final loss: 0.0102


In [ ]:
import json

def evaluate_model(model, tokenizer, eval_dataset, max_new_tokens=512):
    model.eval()
    results = []

    for sample in eval_dataset:
        text = tokenizer.apply_chat_template(
            sample['prompt'],
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors='pt').to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,          # Greedy for deterministic pass@1
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated = tokenizer.decode(
            output_ids[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )

        passed = task_pass_reward(
            [generated], test_script=[sample['test_script']]
        )[0]

        results.append({
            'difficulty': sample['difficulty'],
            'category': sample['category'],
            'passed': bool(passed),
            'snippet': generated[:300],
        })

    total = len(results)
    n_passed = sum(r['passed'] for r in results)
    accuracy = n_passed / total if total else 0.0

    by_diff = {}
    for d in ['easy', 'medium', 'hard']:
        subset = [r for r in results if r['difficulty'] == d]
        if subset:
            by_diff[d] = sum(r['passed'] for r in subset) / len(subset)

    return accuracy, by_diff, results


accuracy, by_diff, detailed = evaluate_model(model, tokenizer, eval_dataset)

print('═══ Evaluation Results ═══')
print(f'Overall pass@1: {accuracy:.1%}  ({sum(r["passed"] for r in detailed)}/{len(detailed)})')
for diff, acc in by_diff.items():
    print(f'  {diff.capitalize():8}: {acc:.1%}')
print()
for i, r in enumerate(detailed):
    status = 'PASS' if r['passed'] else 'FAIL'
    print(f'  [{i+1}] {status} | {r["difficulty"]:6} | {r["category"]}')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


═══ Evaluation Results ═══
Overall pass@1: 25.0%  (1/4)
  Easy    : 0.0%
  Medium  : 0.0%
  Hard    : 50.0%

  [1] FAIL | easy   | data-processing
  [2] FAIL | medium | sysadmin
  [3] PASS | hard   | security
  [4] FAIL | hard   | scripting


In [ ]:
SAVE_PATH = '/content/rlvr_terminalbench2_lora'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

summary = {
    'model': MODEL_ID,
    'benchmark': 'TerminalBench2',
    'framework': 'HuggingFace TRL (GRPO)',
    'eval_pass_at_1': accuracy,
    'by_difficulty': by_diff,
}
with open(f'{SAVE_PATH}/eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Adapter saved to {SAVE_PATH}')
print(json.dumps(summary, indent=2))

Adapter saved to /content/rlvr_terminalbench2_lora
{
  "model": "Qwen/Qwen2.5-7B-Instruct",
  "benchmark": "TerminalBench2",
  "framework": "HuggingFace TRL (GRPO)",
  "eval_pass_at_1": 0.25,
  "by_difficulty": {
    "easy": 0.0,
    "medium": 0.0,
    "hard": 0.5
  }
}


In [ ]:
def run_inference(model, tokenizer, task: str, max_new_tokens=512) -> str:
    prompt = build_prompt(task)
    text = tokenizer.apply_chat_template(
        prompt, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )


NOVEL_TASK = (
    'Show the top 5 processes by CPU usage and save the output to /tmp/top_procs.txt.'
)

print('Task:', NOVEL_TASK)
response = run_inference(model, tokenizer, NOVEL_TASK)
print('\nFull response:')
print(response)
print('\nExtracted answer:')
print(extract_answer(response))

Task: Show the top 5 processes by CPU usage and save the output to /tmp/top_procs.txt.

Full response:
<think>
To achieve this, we need to use the `top` command to get the list of processes sorted by CPU usage and then redirect the output to a file. The `top` command can be used with the `-b` (batch mode) and `-n 1` (run only once) options to avoid interactive prompts. We will then sort the output by CPU usage and take the top 5 processes.
</think>

<answer>
top -b -n 1 | grep "PID" -A 5 | sort -rn -k 9 | head -n 6 > /tmp/top_procs.txt
</answer>

Extracted answer:
top -b -n 1 | grep "PID" -A 5 | sort -rn -k 9 | head -n 6 > /tmp/top_procs.txt
